# Unidad 1: Bases de Datos Orientadas a Documentos
### Procesamiento de Datos (Nivel 4)
**Carrera de Ciencia de Datos** — Modalidad En Línea
**Docente:** Galo Valverde

---

Esta unidad introduce los fundamentos esenciales del procesamiento e ingeniería de datos en la era moderna,
marcando la transición desde el desarrollo de software local hacia las infraestructuras de datos distribuidas.

**Contenido:**
1. El ciclo de vida de los datos: Ingeniería vs. Ciencia de Datos
2. Evolución de la arquitectura de datos (relacional → distribuida)
3. Tipos y estructuras de datos (estructurados, semiestructurados, no estructurados)
4. Formatos de intercambio: CSV, JSON, XML y Apache Parquet
5. Metadatos y diccionarios de datos
6. Adquisición de datos: APIs REST, web scraping, bases de datos, archivos en la nube
7. Fundamentos de NoSQL: SQL vs. Documental
8. MongoDB: colecciones, documentos y operaciones CRUD
9. Integración con Pandas DataFrames


## 1. El Ciclo de Vida de los Datos

El ciclo de vida separa la **infraestructura** del **análisis**:

| Fase | Ingeniería de Datos | Ciencia de Datos |
|---|---|---|
| Enfoque | Construye los cimientos | Extrae el valor |
| Etapas | Adquirir → Transformar → Almacenar | Analizar → Modelar → Predecir |
| Resultado | Flujos escalables de datos limpios | Estadística, ML, decisiones de negocio |

> Esta unidad se centra estrictamente en la **fase de ingeniería**: cómo capturar el caos del mundo real y prepararlo para el análisis.

### Evolución de la arquitectura de datos

- **El pasado — Arquitecturas tradicionales (relacional):** silos rígidos diseñados para transacciones estructuradas. Esquemas inflexibles que colapsan ante el volumen y variedad de los datos web modernos.
- **El presente — Arquitecturas modernas (distribuidas):** diseñadas para la variedad. Integran bases de datos NoSQL, almacenamiento en la nube y procesamiento distribuido. El esquema se adapta a los datos, no al revés.


## 2. El Espectro de la Estructura de Datos

| Tipo | Descripción | Ejemplos |
|---|---|---|
| **No estructurados** | ~80% del mundo real. Texto libre, imágenes, video, audio. Sin esquema predefinido; imposible de consultar con lenguajes tradicionales. | PDFs, fotos, audio |
| **Semiestructurados** | El estándar web. Formatos como JSON y XML. Etiquetas y metadatos internos separan elementos semánticos. Esquema flexible y auto-descriptivo. | JSON, XML |
| **Estructurados** | Bases relacionales: tablas de filas y columnas. Esquema estricto y predefinido antes de la inserción. | SQL, CSV |

### Anatomía de los formatos de datos

- **CSV** — Ligero, tabular. Ideal para exportar datos simples, pero incapaz de manejar jerarquías o relaciones complejas.
- **JSON** — Legible por humanos y máquinas. Estructuras anidadas (árbol). Es el formato nativo de las bases de datos orientadas a documentos.
- **Parquet** — Formato binario columnar. Optimizado para cargas analíticas masivas (Big Data), reduciendo drásticamente el espacio y tiempo de lectura.


In [1]:
# Ejemplo práctico: el mismo registro en CSV vs. JSON vs. Parquet
import pandas as pd
import json

registro = {
    "Estudiante": {
        "Nombre": "Juan Perez",
        "ID": "CC3380",
        "Especialidad": "CC"
    }
}

# --- CSV (aplanado, no soporta anidamiento nativo) ---
df_plano = pd.DataFrame([{
    "Nombre": registro["Estudiante"]["Nombre"],
    "NumEstudiante": registro["Estudiante"]["ID"],
    "Especialidad": registro["Estudiante"]["Especialidad"]
}])
df_plano.to_csv("estudiante.csv", index=False)
print("--- CSV ---")
print(df_plano.to_csv(index=False))

# --- JSON (estructura anidada nativa) ---
with open("estudiante.json", "w") as f:
    json.dump(registro, f, indent=2)
print("--- JSON ---")
print(json.dumps(registro, indent=2))

# --- Parquet (columnar, requiere estructura tabular) ---
df_plano.to_parquet("estudiante.parquet", index=False)
print("--- Parquet ---")
print(pd.read_parquet("estudiante.parquet"))


--- CSV ---
Nombre,NumEstudiante,Especialidad
Juan Perez,CC3380,CC

--- JSON ---
{
  "Estudiante": {
    "Nombre": "Juan Perez",
    "ID": "CC3380",
    "Especialidad": "CC"
  }
}
--- Parquet ---
       Nombre NumEstudiante Especialidad
0  Juan Perez        CC3380           CC


In [2]:
# Comparación de tamaño en disco (ilustrativo con un dataset más grande)
import numpy as np
import os

n = 50_000
df_grande = pd.DataFrame({
    "id": range(n),
    "categoria": np.random.choice(["A", "B", "C", "D"], n),
    "valor": np.random.randn(n),
    "fecha": pd.date_range("2020-01-01", periods=n, freq="h")
})

df_grande.to_csv("grande.csv", index=False)
df_grande.to_parquet("grande.parquet", index=False)
df_grande.to_json("grande.json", orient="records")

for archivo in ["grande.csv", "grande.parquet", "grande.json"]:
    size_kb = os.path.getsize(archivo) / 1024
    print(f"{archivo:20s} -> {size_kb:,.1f} KB")


grande.csv           -> 2,315.0 KB
grande.parquet       -> 1,221.8 KB
grande.json          -> 3,523.8 KB


/tmp/ipykernel_537/4105149796.py:15: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_grande.to_json("grande.json", orient="records")


## 3. Metadatos: El ADN de tus Datos

> "Los datos sin contexto son solo ruido."

**¿Qué son los metadatos?**
Datos sobre los datos: describen el origen, formato, tamaño, autor y fecha de creación de un conjunto de datos.
En formatos semiestructurados (JSON), las claves viajan incrustadas con el dato mismo.

**El diccionario de datos**
Un repositorio centralizado que define el significado de cada campo (ej. ¿el campo `temp` está en Celsius o Fahrenheit?).
Es fundamental para evitar errores de interpretación en el análisis.


In [3]:
# Ejemplo: diccionario de datos como estructura documentada
diccionario_de_datos = [
    {"campo": "id",         "tipo": "int",      "unidad": None,   "descripcion": "Identificador único del registro"},
    {"campo": "categoria",  "tipo": "string",   "unidad": None,   "descripcion": "Categoría asignada (A-D)"},
    {"campo": "valor",      "tipo": "float",    "unidad": "z-score", "descripcion": "Valor normalizado de la medición"},
    {"campo": "fecha",      "tipo": "datetime", "unidad": "UTC",  "descripcion": "Marca temporal de captura del dato"},
]

df_diccionario = pd.DataFrame(diccionario_de_datos)
df_diccionario


,campo,tipo,unidad,descripcion
0,id,int,NaN,Identificador único del registro
1,categoria,string,NaN,Categoría asignada (A-D)
2,valor,float,z-score,Valor normalizado de la medición
3,fecha,datetime,UTC,Marca temporal de captura del dato


## 4. Adquisición: Capturando Datos en su Hábitat Natural

Cuatro vías principales de ingesta:

| Fuente | Descripción |
|---|---|
| **Bases de datos** | Conexiones directas para extraer información transaccional existente |
| **APIs REST** | Comunicación estandarizada servidor-a-servidor; devuelve payloads en JSON |
| **Archivos** | Ingesta por lotes (batch) desde AWS S3, Google Cloud, o sistemas locales |
| **Scraping** | Extracción de datos incrustados en código HTML de páginas web |

### APIs REST vs. Web Scraping

- **La vía oficial (API REST):** sistemas diseñados para ser consumidos por máquinas. Responden a peticiones entregando datos limpios y estructurados, listos para almacenar.
- **La vía no oficial (Web Scraping):** cuando no hay API, se parsea el árbol DOM (HTML) usando librerías de Python. Convierte información diseñada para el ojo humano en formatos legibles por máquina.


In [4]:
# 4.1 Consumo de una API REST pública (ejemplo con JSONPlaceholder)
import requests

try:
    resp = requests.get("https://jsonplaceholder.typicode.com/users", timeout=5)
    resp.raise_for_status()
    usuarios = resp.json()
    df_usuarios = pd.DataFrame(usuarios)[["id", "name", "email", "company"]]
    df_usuarios["empresa"] = df_usuarios["company"].apply(lambda c: c["name"])
    df_usuarios = df_usuarios.drop(columns="company")
    print(df_usuarios.head())
except requests.exceptions.RequestException as e:
    print("Sin conexión a internet en este entorno. Error:", e)
    print("En un entorno con salida a internet, este bloque descarga 10 usuarios de ejemplo desde la API REST.")


Sin conexión a internet en este entorno. Error: 403 Client Error: Forbidden for url: https://jsonplaceholder.typicode.com/users
En un entorno con salida a internet, este bloque descarga 10 usuarios de ejemplo desde la API REST.


In [5]:
# 4.2 Conexión a una base de datos relacional (SQLite como ejemplo local)
import sqlite3

conn = sqlite3.connect("empresa.db")
cur = conn.cursor()
cur.execute('''
CREATE TABLE IF NOT EXISTS empleados (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    departamento TEXT,
    salario REAL
)
''')
cur.executemany(
    "INSERT INTO empleados (nombre, departamento, salario) VALUES (?, ?, ?)",
    [("Ana Torres", "TI", 1450.0),
     ("Luis Vera", "Finanzas", 1600.0),
     ("Maria Paz", "RR.HH.", 1300.0)]
)
conn.commit()

# Extracción programática hacia un DataFrame
df_empleados = pd.read_sql_query("SELECT * FROM empleados", conn)
conn.close()
df_empleados


,id,nombre,departamento,salario
0,1,Ana Torres,TI,1450.0
1,2,Luis Vera,Finanzas,1600.0
2,3,Maria Paz,RR.HH.,1300.0


In [6]:
# 4.3 Web scraping básico con BeautifulSoup
from bs4 import BeautifulSoup

html_ejemplo = '''
<html>
  <body>
    <div class="curso">
      <span class="titulo">Procesamiento de Datos</span>
      <span class="nivel">Nivel 4</span>
    </div>
    <div class="curso">
      <span class="titulo">Sistemas Embebidos</span>
      <span class="nivel">Nivel 5</span>
    </div>
  </body>
</html>
'''

soup = BeautifulSoup(html_ejemplo, "html.parser")
cursos = []
for div in soup.find_all("div", class_="curso"):
    cursos.append({
        "titulo": div.find("span", class_="titulo").text,
        "nivel": div.find("span", class_="nivel").text
    })

pd.DataFrame(cursos)


,titulo,nivel
0,Procesamiento de Datos,Nivel 4
1,Sistemas Embebidos,Nivel 5


## 5. La Matriz de Diagnóstico: SQL vs. NoSQL

| Dimensión | Relacional (SQL) | Documental (NoSQL) |
|---|---|---|
| **Esquema** | Rígido (debe definirse antes de insertar) | Dinámico/flexible (cada documento puede tener una estructura distinta) |
| **Estructura** | Tablas planas interconectadas por claves foráneas (joins) | Documentos anidados, frecuentemente JSON/BSON, autocontenidos |
| **Escalabilidad** | Vertical (requiere servidores más potentes) | Horizontal (se distribuye fácilmente entre múltiples clústeres) |
| **Uso ideal** | Transacciones (OLTP), integridad referencial estricta | Catálogos de e-commerce, perfiles web, datos de APIs |


## 6. La Anatomía de MongoDB

Dejando atrás las tablas bidimensionales:

- **Clúster** — Servidores distribuidos.
- **Base de datos** — Ej. `E-commerce`.
- **Colección** — Reemplaza a las tablas. Agrupación lógica de documentos; a diferencia de las tablas SQL, una colección **no impone un esquema** a los documentos que contiene.
- **Documento BSON** — La unidad básica de datos (Binary JSON). Permite almacenar listas y sub-documentos dentro de un solo registro, eliminando joins complejos.

### Operaciones CRUD: el lenguaje de los datos

```python
# Crear (Insert)
db.usuarios.insert_one({'nombre': 'Ana', 'edad': 28})

# Leer (Find)
db.usuarios.find({'edad': {'$gt': 25}})

# Actualizar (Update)
db.usuarios.update_one({'nombre': 'Ana'}, {'$set': {'rol': 'Admin'}})

# Borrar (Delete)
db.usuarios.delete_one({'nombre': 'Ana'})
```

> Operar sobre documentos JSON se siente como programar orientado a objetos, no como redactar consultas de texto planas.

**Nota de entorno:** este notebook usa `mongomock` para simular un servidor MongoDB sin necesidad de instalación,
de modo que el código CRUD se pueda ejecutar tal cual contra un servidor real (`pymongo.MongoClient`) simplemente
cambiando la línea de conexión.


In [7]:
# Instalar dependencias necesarias (ejecutar una sola vez)
# !pip install pymongo mongomock --quiet


In [8]:
# 6.1 Conexión (simulada con mongomock; en producción: pymongo.MongoClient("mongodb://localhost:27017/"))
import mongomock
# from pymongo import MongoClient  # <- usar esta línea con un servidor MongoDB real

client = mongomock.MongoClient()
db = client["ecommerce"]
usuarios = db["usuarios"]
print("Conectado a la base de datos:", db.name)


Conectado a la base de datos: ecommerce


In [9]:
# 6.2 CREATE (Insert) — esquema flexible: cada documento puede variar
usuarios.insert_many([
    {"nombre": "Ana",   "edad": 28, "rol": "Cliente", "direccion": {"ciudad": "Guayaquil", "pais": "EC"}},
    {"nombre": "Luis",  "edad": 34, "rol": "Cliente", "compras": ["laptop", "mouse"]},
    {"nombre": "Maria", "edad": 22, "rol": "Cliente"},
])
print(f"Documentos insertados: {usuarios.count_documents({})}")


Documentos insertados: 3


In [10]:
# 6.3 READ (Find) — consultas con operadores
print("Usuarios con edad > 25:")
for doc in usuarios.find({"edad": {"$gt": 25}}):
    print(doc)


Usuarios con edad > 25:
{'nombre': 'Ana', 'edad': 28, 'rol': 'Cliente', 'direccion': {'ciudad': 'Guayaquil', 'pais': 'EC'}, '_id': ObjectId(4818ad31-a5b9-11f1-8390-8304be10e346)}
{'nombre': 'Luis', 'edad': 34, 'rol': 'Cliente', 'compras': ['laptop', 'mouse'], '_id': ObjectId(4818bb0a-a5b9-11f1-8897-8304be10e346)}


In [11]:
# 6.4 UPDATE — modificar un documento existente
usuarios.update_one({"nombre": "Ana"}, {"$set": {"rol": "Admin"}})
print(usuarios.find_one({"nombre": "Ana"}))


{'nombre': 'Ana', 'edad': 28, 'rol': 'Admin', 'direccion': {'ciudad': 'Guayaquil', 'pais': 'EC'}, '_id': ObjectId(4818ad31-a5b9-11f1-8390-8304be10e346)}


In [12]:
# 6.5 DELETE — eliminar un documento
usuarios.delete_one({"nombre": "Maria"})
print(f"Documentos restantes: {usuarios.count_documents({})}")


Documentos restantes: 2


## 7. Pandas: El Puente Hacia la Analítica

Flujo típico de integración MongoDB → Pandas:

1. **Conexión y lectura** — Python extrae los documentos JSON desde MongoDB mediante cursores.
2. **Aplanamiento (flattening)** — las estructuras anidadas complejas de JSON se transforman o normalizan.
3. **Preparación (EDA)** — los datos ingresan a un DataFrame bidimensional de Pandas para limpieza, manejo de nulos y manipulación vectorial.


In [13]:
# 7.1 De MongoDB a un DataFrame plano con pandas.json_normalize
cursor = list(usuarios.find({}, {"_id": 0}))  # excluimos el _id de BSON para simplificar
df_usuarios_mongo = pd.json_normalize(cursor, sep="_")
df_usuarios_mongo


,nombre,edad,rol,direccion_ciudad,direccion_pais,compras
0,Ana,28,Admin,Guayaquil,EC,NaN
1,Luis,34,Cliente,NaN,NaN,"[laptop, mouse]"


In [14]:
# 7.2 Preparación (EDA) básica sobre el DataFrame resultante
print("Valores nulos por columna:")
print(df_usuarios_mongo.isna().sum())
print()
print("Resumen estadístico:")
df_usuarios_mongo.describe(include="all")


Valores nulos por columna:
nombre              0
edad                0
rol                 0
direccion_ciudad    1
direccion_pais      1
compras             1
dtype: int64

Resumen estadístico:


,nombre,edad,rol,direccion_ciudad,direccion_pais,compras
count,2,2.000000,2,1,1,1
unique,2,NaN,2,1,1,1
top,Ana,NaN,Admin,Guayaquil,EC,"[laptop, mouse]"
freq,1,NaN,1,1,1,1
mean,NaN,31.000000,NaN,NaN,NaN,NaN
std,NaN,4.242641,NaN,NaN,NaN,NaN
min,NaN,28.000000,NaN,NaN,NaN,NaN
25%,NaN,29.500000,NaN,NaN,NaN,NaN
50%,NaN,31.000000,NaN,NaN,NaN,NaN
75%,NaN,32.500000,NaN,NaN,NaN,NaN


## 8. El Pipeline Definitivo (La Refinería en Acción)

1. **Captura** — Scraping o API REST extrae la data.
2. **Formateo** — Se preserva como JSON por su flexibilidad.
3. **Almacenamiento** — Se inyecta en MongoDB sin requerir un esquema rígido previo.
4. **Preparación** — Se exporta a Pandas para su análisis y modelado.

> Misión del Ingeniero de Datos completada.


In [15]:
# Pipeline completo de extremo a extremo (síntesis de la unidad)
import json

# 1. Captura (simulada — en un caso real sería requests.get(...) o scraping)
datos_capturados = [
    {"producto": "Laptop X1", "precio": 850.0, "stock": 12, "categoria": "Tecnología"},
    {"producto": "Mouse Inalámbrico", "precio": 18.5, "stock": 120, "categoria": "Accesorios"},
    {"producto": "Monitor 24\"", "precio": 210.0, "stock": 30, "categoria": "Tecnología"},
]

# 2. Formateo — ya está en JSON (lista de diccionarios de Python)
print("2. Formateo (JSON):")
print(json.dumps(datos_capturados, indent=2, ensure_ascii=False))

# 3. Almacenamiento en MongoDB (sin esquema rígido)
productos = db["productos"]
productos.insert_many(datos_capturados)
print(f"\n3. Almacenamiento: {productos.count_documents({})} documentos en la colección 'productos'")

# 4. Preparación — exportar a Pandas para análisis
df_productos = pd.json_normalize(list(productos.find({}, {"_id": 0})))
print("\n4. Preparación (DataFrame listo para análisis):")
df_productos


2. Formateo (JSON):
[
  {
    "producto": "Laptop X1",
    "precio": 850.0,
    "stock": 12,
    "categoria": "Tecnología"
  },
  {
    "producto": "Mouse Inalámbrico",
    "precio": 18.5,
    "stock": 120,
    "categoria": "Accesorios"
  },
  {
    "producto": "Monitor 24\"",
    "precio": 210.0,
    "stock": 30,
    "categoria": "Tecnología"
  }
]

3. Almacenamiento: 3 documentos en la colección 'productos'

4. Preparación (DataFrame listo para análisis):


,producto,precio,stock,categoria
0,Laptop X1,850.0,12,Tecnología
1,Mouse Inalámbrico,18.5,120,Accesorios
2,"Monitor 24""",210.0,30,Tecnología


## 9. Referencias y Enlaces de Consulta

**Enlaces oficiales:**
- Manual Oficial de MongoDB (CRUD): https://www.mongodb.com/docs/manual/crud/
- Documentación de Pandas de Lectura/Escritura: https://pandas.pydata.org/docs/user_guide/io.html

**Referencias bibliográficas (APA):**
- McKinney, W. (2022). *Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media.
- Silberschatz, A., Korth, H. F., & Sudarshan, S. (2020). *Database System Concepts* (7th ed.). McGraw-Hill.
- VanderPlas, J. (2023). *Python Data Science Handbook: Essential Tools for Working with Data* (2nd ed.). O'Reilly Media.
